# Advanced Usage

This notebook covers:
- Gene data access and gene-constrained simulation
- Graph set operations (union, intersection, difference)
- Bayesian posterior updates
- ML feature extraction
- Logging and verbose control
- CLI usage from Python

In [ ]:
from LZGraphs import LZGraph, set_log_level
import numpy as np
import csv

In [ ]:
sequences, v_genes, j_genes = [], [], []
with open('ExampleData3.csv') as f:
    for row in csv.DictReader(f):
        sequences.append(row['cdr3_amino_acid'])
        v_genes.append(row['V'])
        j_genes.append(row['J'])

graph = LZGraph(sequences, variant='aap', v_genes=v_genes, j_genes=j_genes)
print(graph)

## 1. Gene Data

When V/J gene annotations are provided during construction, the graph stores marginal and joint gene distributions, plus per-edge gene counts.

In [ ]:
# V gene marginals (sorted by frequency)
print('Top 10 V genes:')
for name, prob in sorted(graph.v_marginals.items(), key=lambda x: -x[1])[:10]:
    print(f'  {name:20s}  {prob:.4f}')

In [ ]:
# J gene marginals
print('J genes:')
for name, prob in sorted(graph.j_marginals.items(), key=lambda x: -x[1]):
    print(f'  {name:20s}  {prob:.4f}')

In [ ]:
# Joint VJ distribution (top 5 pairs)
print('Top 5 VJ pairs:')
vj = sorted(graph.vj_distribution, key=lambda x: -x['prob'])
for pair in vj[:5]:
    print(f'  {pair["v"]:20s} + {pair["j"]:15s}  P = {pair["prob"]:.4f}')

In [ ]:
# Gene-constrained simulation: sample from joint VJ distribution
result = graph.simulate(5, seed=42, sample_genes=True)
for seq, v, j in zip(result.sequences, result.v_genes, result.j_genes):
    print(f'{seq:25s}  V={v}  J={j}')

In [ ]:
# Constrain to a specific V gene
top_v = sorted(graph.v_marginals.items(), key=lambda x: -x[1])[0][0]
result = graph.simulate(5, seed=42, v_gene=top_v)
print(f'Constrained to V={top_v}:')
for seq, v in zip(result.sequences, result.v_genes):
    print(f'  {seq:25s}  V={v}')

## 2. Graph Set Operations

LZGraphs supports set algebra on graphs: union, intersection, difference, and weighted merge. These operate on edge counts and produce new graphs.

In [ ]:
# Split into two repertoires
g_a = LZGraph(sequences[:2500], variant='aap')
g_b = LZGraph(sequences[2500:], variant='aap')
print(f'A: {g_a}')
print(f'B: {g_b}')

In [ ]:
# Union: combine both repertoires (sum edge counts)
g_union = g_a | g_b    # or: g_a.union(g_b)
print(f'A | B: {g_union}')

# Intersection: keep only shared structure
g_inter = g_a & g_b    # or: g_a.intersection(g_b)
print(f'A & B: {g_inter}')

# Difference: what's unique to A
g_diff = g_a - g_b     # or: g_a.difference(g_b)
print(f'A - B: {g_diff}')

In [ ]:
# Weighted merge
g_weighted = g_a.weighted_merge(g_b, alpha=0.7, beta=0.3)
print(f'0.7*A + 0.3*B: {g_weighted}')

## 3. Bayesian Posterior

Update a graph's edge weights given new observed sequences. The `kappa` parameter controls the strength of the prior.

In [ ]:
# Start with a prior built from first 1000 sequences
prior = LZGraph(sequences[:1000], variant='aap')

# Update with 500 new sequences
posterior = prior.posterior(sequences[1000:1500], kappa=10.0)

# Compare probabilities: pick a sequence from the prior's training data
seq = sequences[0]
print(f'Sequence: {seq}')
print(f'Prior log P:     {prior.lzpgen(seq):.4f}')
print(f'Posterior log P: {posterior.lzpgen(seq):.4f}')
print(f'Posterior has more data → probabilities shift.')


## 4. ML Feature Extraction

Extract numerical feature vectors from graphs for use in machine learning pipelines.

In [ ]:
# 15-element statistics vector
stats = graph.feature_stats()
labels = ['nodes', 'edges', 'n_initial', 'n_terminal',
          'D(0)', 'D(0.5)', 'D(1)', 'D(2)', 'D(5)',
          'entropy', 'dyn_range', 'mean_stop', 'std_stop',
          'max_out_degree', 'uniformity']
for label, val in zip(labels, stats):
    print(f'  {label:18s} = {val:.4f}')

In [ ]:
# Reference-aligned feature vector
# Project g_b into g_a's node space
aligned = g_a.feature_aligned(g_b)
print(f'Aligned vector: shape={aligned.shape}, nonzero={np.count_nonzero(aligned)}/{len(aligned)}')

In [ ]:
# Mass profile: probability mass by sequence position
profile = graph.feature_mass_profile(max_pos=30)
print(f'Mass profile: shape={profile.shape}, sum={profile.sum():.4f}')
print(f'Peak position: {np.argmax(profile)}')

## 5. Logging

Control the verbose output from the C library.

In [ ]:
# Enable INFO-level logging
set_log_level('info')
g = LZGraph(sequences[:100], variant='aap')

# Disable
set_log_level('none')

In [ ]:
# Custom callback (e.g., collect messages)
from LZGraphs import set_log_callback

messages = []
set_log_callback(lambda level, msg: messages.append(msg), level='debug')

g = LZGraph(sequences[:100], variant='aap')
_ = g.simulate(10, seed=42)

print(f'Captured {len(messages)} log messages:')
for m in messages:
    print(f'  {m}')

# Reset
set_log_callback(None)

## 6. Dynamic Range

The dynamic range measures how many orders of magnitude separate the most and least probable sequences.

In [ ]:
dr = graph.pgen_dynamic_range()
print(f'Dynamic range: {dr:.2f} orders of magnitude')

detail = graph.pgen_dynamic_range_detail()
print(f'Most probable:  log P = {detail["max_log_prob"]:.4f}')
print(f'Least probable: log P = {detail["min_log_prob"]:.4f}')